# The Goertzel Algorithm

The Goertzel Algorithm is an efficient method for computing a **single frequency bin** of the DFT. Where a full FFT costs O(N log N) to evaluate all N bins, Goertzel costs O(N) for one target frequency. It's ideal when you only need to detect or measure a known tone — DTMF detection is the classic example.

This notebook walks through the algorithm from first principles, then visualises what each stage is doing.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

## 1. What is a DFT bin?

The Discrete Fourier Transform at bin $k$ is:

$$X[k] = \sum_{n=0}^{N-1} x[n] \cdot e^{-j 2\pi k n / N}$$

This is a **correlation** of the signal $x[n]$ with a complex exponential (a spinning phasor) at normalised frequency $\omega_k = 2\pi k / N$.

Computing this directly requires $N$ complex multiplications per bin. Goertzel recasts it as a **recursive filter** so we avoid the multiplications entirely during the accumulation loop.

## 2. Derivation — from DFT to a difference equation

Multiply $X[k]$ by $1 = e^{j 2\pi k N / N} = W_N^{-kN}$ (this equals 1 because $e^{j2\pi k} = 1$ for integer $k$):

$$X[k] = W_N^{-kN} \sum_{n=0}^{N-1} x[n] W_N^{kn} = \sum_{n=0}^{N-1} x[n] W_N^{-k(N-n)}$$

where $W_N = e^{j2\pi/N}$ and $W_N^{-k} = e^{-j2\pi k/N}$.

Define an intermediate sequence:

$$y_k[n] = \sum_{m=0}^{n} x[m] \cdot W_N^{-k(n-m)}$$

This satisfies the **recurrence**:

$$\boxed{y_k[n] = x[n] + W_N^{-k} \cdot y_k[n-1]}$$

with $y_k[-1] = 0$, and the final answer is $X[k] = y_k[N-1]$.

That recurrence *is* Goertzel's algorithm. But we can make it cheaper.

## 3. Eliminating the complex multiply

The multiplier $W_N^{-k} = e^{-j2\pi k/N}$ is complex. We can split $y_k[n]$ into a two-stage filter:

**Stage 1 — real-coefficient IIR (run for all N samples):**

$$s[n] = x[n] + 2\cos(\omega_k)\, s[n-1] - s[n-2]$$

with $s[-1] = s[-2] = 0$ and $\omega_k = 2\pi k / N$.

**Stage 2 — one complex multiply at the very end (sample N only):**

$$X[k] = s[N-1] - W_N^k \cdot s[N-2]$$

The key insight: the IIR loop uses **only real arithmetic** — one multiply and two additions per sample. The single complex multiply happens just once at the end.

> **Cost**: $N$ real multiplications + 1 complex multiplication, versus $N$ complex multiplications in the naive DFT.

## 4. Reference implementation

In [ ]:
def goertzel(x, k, N=None):
    """Goertzel algorithm — returns complex DFT value at bin k."""
    if N is None:
        N = len(x)
    omega = 2 * np.pi * k / N
    coeff = 2 * np.cos(omega)

    s_prev2 = 0.0  # s[n-2]
    s_prev1 = 0.0  # s[n-1]

    for sample in x:
        s = sample + coeff * s_prev1 - s_prev2
        s_prev2 = s_prev1
        s_prev1 = s

    # Final single complex multiply
    X_k = s_prev1 - np.exp(-1j * omega) * s_prev2
    return X_k


def goertzel_magnitude_sq(x, k, N=None):
    """Return |X[k]|² using only real arithmetic throughout."""
    if N is None:
        N = len(x)
    omega = 2 * np.pi * k / N
    coeff = 2 * np.cos(omega)

    s_prev2 = 0.0
    s_prev1 = 0.0

    for sample in x:
        s = sample + coeff * s_prev1 - s_prev2
        s_prev2 = s_prev1
        s_prev1 = s

    # |X[k]|² = s1² + s2² - coeff·s1·s2  (fully real)
    return s_prev1**2 + s_prev2**2 - coeff * s_prev1 * s_prev2

### Verify against NumPy's FFT

In [ ]:
rng = np.random.default_rng(0)
N = 64
x = rng.standard_normal(N)

fft_result = np.fft.fft(x)

print(f"{'bin':>4}  {'FFT':>30}  {'Goertzel':>30}  {'|error|':>12}")
print("-" * 82)
for k in [0, 1, 7, 16, 31, 63]:
    g = goertzel(x, k, N)
    err = abs(fft_result[k] - g)
    print(f"{k:>4}  {fft_result[k]:>30.6f}  {g:>30.6f}  {err:>12.2e}")

## 5. Visualising the IIR state evolution

Below we plot $s[n]$, $s[n-1]$, and the running contribution $s[n]^2 + s[n-1]^2 - \text{coeff}\cdot s[n]s[n-1]$ (an approximation of the accumulated power) as we feed a pure tone into the filter.

In [ ]:
def goertzel_trace(x, k, N=None):
    """Return per-sample state history for visualisation."""
    if N is None:
        N = len(x)
    omega = 2 * np.pi * k / N
    coeff = 2 * np.cos(omega)

    s1_hist, s2_hist, power_hist = [], [], []
    s1, s2 = 0.0, 0.0

    for sample in x:
        s = sample + coeff * s1 - s2
        s2 = s1
        s1 = s
        s1_hist.append(s1)
        s2_hist.append(s2)
        power_hist.append(s1**2 + s2**2 - coeff * s1 * s2)

    return np.array(s1_hist), np.array(s2_hist), np.array(power_hist)


N = 128
target_k = 8          # bin we want to detect
off_target_k = 12     # a different bin for comparison
t = np.arange(N)

# Pure tone at target_k
x_target = np.sin(2 * np.pi * target_k * t / N)

s1_on,  s2_on,  power_on  = goertzel_trace(x_target, target_k, N)
s1_off, s2_off, power_off = goertzel_trace(x_target, off_target_k, N)

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(t, x_target, color='steelblue', lw=1.5)
axes[0].set_ylabel('Amplitude')
axes[0].set_title(f'Input signal — pure tone at bin {target_k}')

axes[1].plot(t, s1_on,  label=f's[n] at bin {target_k} (matched)',      color='seagreen', lw=1.5)
axes[1].plot(t, s1_off, label=f's[n] at bin {off_target_k} (mismatched)', color='tomato',   lw=1.5, alpha=0.8)
axes[1].set_ylabel('Filter state s[n]')
axes[1].set_title('IIR state grows when filter frequency matches input')
axes[1].legend()

axes[2].plot(t, power_on,  label=f'Power at bin {target_k}',      color='seagreen', lw=1.5)
axes[2].plot(t, power_off, label=f'Power at bin {off_target_k}',  color='tomato',   lw=1.5, alpha=0.8)
axes[2].set_ylabel('|X[k]|² estimate')
axes[2].set_xlabel('Sample index n')
axes[2].set_title('Accumulated power — only the matched bin resonates')
axes[2].legend()

fig.tight_layout()
plt.show()

## 6. Frequency selectivity — the filter's frequency response

The Goertzel IIR has a **second-order resonator** transfer function:

$$H(z) = \frac{1}{1 - 2\cos(\omega_k)z^{-1} + z^{-2}}$$

It has a pole pair on the unit circle at $\pm\omega_k$. The longer the block size $N$, the narrower its effective bandwidth (because we only care about the accumulated output after exactly $N$ steps).

Below we plot the magnitude response for several target frequencies.

In [ ]:
def goertzel_freq_response(k, N, n_freqs=1000):
    omega_k = 2 * np.pi * k / N
    coeff = 2 * np.cos(omega_k)
    freqs = np.linspace(0, np.pi, n_freqs)
    z = np.exp(1j * freqs)
    H = 1 / (1 - coeff * z**-1 + z**-2)
    return freqs, np.abs(H)

N = 64
target_bins = [4, 8, 16, 24]
colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(target_bins)))

fig, ax = plt.subplots(figsize=(12, 4))
for k, color in zip(target_bins, colors):
    freqs, H = goertzel_freq_response(k, N)
    ax.plot(freqs / np.pi, 20 * np.log10(H + 1e-12), color=color,
            label=f'bin k={k}  (f = {k/N:.3f}·fs)')

ax.set_xlabel('Normalised frequency (× π rad/sample)')
ax.set_ylabel('|H(f)| dB')
ax.set_title(f'Goertzel resonator frequency response (N={N})')
ax.set_ylim(-30, 50)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

## 7. Block size N and frequency resolution

The frequency resolution (bin width) is $\Delta f = f_s / N$. A larger block gives finer resolution but more latency. Here we show how increasing $N$ sharpens the detection of a target tone when a nearby interferer is present.

In [ ]:
fs = 8000          # sample rate (Hz)
f_target = 697     # DTMF row tone (Hz)
f_interferer = 770 # nearby tone
block_sizes = [64, 128, 256, 512]

fig, axes = plt.subplots(1, len(block_sizes), figsize=(15, 4), sharey=True)

for ax, N in zip(axes, block_sizes):
    t = np.arange(N) / fs
    x = np.sin(2 * np.pi * f_target * t) + 0.5 * np.sin(2 * np.pi * f_interferer * t)

    # Evaluate Goertzel across all bins
    mags = np.array([abs(goertzel(x, k, N)) for k in range(N // 2)])
    bin_freqs = np.arange(N // 2) * fs / N

    ax.plot(bin_freqs, mags, lw=1.5, color='steelblue')
    ax.axvline(f_target, color='seagreen', ls='--', lw=1.5, label=f'{f_target} Hz (target)')
    ax.axvline(f_interferer, color='tomato', ls='--', lw=1.5, label=f'{f_interferer} Hz (interferer)')
    ax.set_title(f'N = {N}\nΔf = {fs/N:.1f} Hz')
    ax.set_xlabel('Frequency (Hz)')
    ax.set_xlim(500, 1000)

axes[0].set_ylabel('|X[k]|')
axes[0].legend(fontsize=8)
fig.suptitle('Effect of block size N on frequency resolution', y=1.02)
plt.tight_layout()
plt.show()

## 8. DTMF tone detection — a practical example

DTMF (telephone keypad tones) encodes each key as a pair of frequencies:

| | 1209 Hz | 1336 Hz | 1477 Hz | 1633 Hz |
|---|---------|---------|---------|---------|
| **697 Hz** | 1 | 2 | 3 | A |
| **770 Hz** | 4 | 5 | 6 | B |
| **852 Hz** | 7 | 8 | 9 | C |
| **941 Hz** | \* | 0 | \# | D |

We run Goertzel at each of the 8 standard frequencies and pick the highest-power row + column.

In [ ]:
DTMF_ROWS = [697, 770, 852, 941]
DTMF_COLS = [1209, 1336, 1477, 1633]
DTMF_KEYS = [
    ['1', '2', '3', 'A'],
    ['4', '5', '6', 'B'],
    ['7', '8', '9', 'C'],
    ['*', '0', '#', 'D'],
]

def detect_dtmf(x, fs, N=None):
    if N is None:
        N = len(x)
    freqs = DTMF_ROWS + DTMF_COLS
    bins = [round(f * N / fs) for f in freqs]
    powers = [goertzel_magnitude_sq(x, k, N) for k in bins]

    row_powers = powers[:4]
    col_powers = powers[4:]
    row = np.argmax(row_powers)
    col = np.argmax(col_powers)
    return DTMF_KEYS[row][col], row_powers, col_powers


fs = 8000
N = 205   # standard DTMF block size at 8 kHz ≈ 25.6 ms
t = np.arange(N) / fs

test_keys = ['5', '3', 'A', '#']

fig, axes = plt.subplots(2, 2, figsize=(13, 7))
axes = axes.ravel()

for ax, key in zip(axes, test_keys):
    # Find row/col frequencies for this key
    for r, row in enumerate(DTMF_KEYS):
        if key in row:
            f_row = DTMF_ROWS[r]
            f_col = DTMF_COLS[row.index(key)]
            break

    x = (np.sin(2 * np.pi * f_row * t)
         + np.sin(2 * np.pi * f_col * t)
         + 0.05 * np.random.default_rng(42).standard_normal(N))

    detected, row_pows, col_pows = detect_dtmf(x, fs, N)

    all_freqs = DTMF_ROWS + DTMF_COLS
    all_pows  = list(row_pows) + list(col_pows)
    colors = ['seagreen' if f in (f_row, f_col) else 'steelblue' for f in all_freqs]

    ax.bar([str(f) for f in all_freqs], all_pows, color=colors)
    ax.set_title(f'Key "{key}"  →  detected "{detected}"  ({'✓' if detected == key else '✗'})')
    ax.set_xlabel('Goertzel frequency (Hz)')
    ax.set_ylabel('Power |X[k]|²')
    ax.tick_params(axis='x', rotation=45)

fig.suptitle('DTMF detection via Goertzel (green bars = expected tones)', y=1.01)
plt.tight_layout()
plt.show()

## 9. Computational cost comparison

When you need $M$ specific bins out of $N$ total, Goertzel beats FFT when $M < N / \log_2 N$.

In [ ]:
Ns = np.logspace(4, 12, 200, base=2)  # block sizes from 16 to 4096

# Real multiplications per output bin
cost_goertzel = Ns          # N real mults per bin
cost_fft_per_bin = (Ns * np.log2(Ns)) / Ns  # total FFT cost ÷ N bins → cost per bin
cost_fft_total = Ns * np.log2(Ns)           # total FFT cost (all bins)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: cost per bin
axes[0].plot(Ns, cost_goertzel,   lw=2, color='seagreen', label='Goertzel (1 bin)')
axes[0].plot(Ns, cost_fft_total,  lw=2, color='steelblue', label='FFT (all N bins)')
axes[0].plot(Ns, cost_fft_per_bin * Ns * 0 + Ns, lw=0)  # placeholder
axes[0].set_xscale('log', base=2)
axes[0].set_yscale('log')
axes[0].set_xlabel('Block size N')
axes[0].set_ylabel('Operations (real mults)')
axes[0].set_title('Goertzel vs FFT — absolute cost')
axes[0].legend()

# Right: breakeven number of bins
breakeven = np.log2(Ns)  # M bins where Goertzel·M = FFT total cost
axes[1].plot(Ns, breakeven, lw=2, color='darkorchid')
axes[1].axhline(8, color='tomato', ls='--', lw=1.5, label='8 DTMF bins')
axes[1].set_xscale('log', base=2)
axes[1].set_xlabel('Block size N')
axes[1].set_ylabel('Break-even M (number of bins)')
axes[1].set_title('Use Goertzel if you need fewer than log₂(N) bins')
axes[1].legend()

plt.tight_layout()
plt.show()

print("DTMF uses 8 bins. Break-even at N:")
for N in [64, 128, 205, 256, 512]:
    be = np.log2(N)
    winner = 'Goertzel' if 8 < be else 'FFT'
    print(f"  N={N:4d}  log₂(N)={be:.1f}  →  {winner} is cheaper")

## 10. Summary

| Property | Detail |
|---|---|
| **What it computes** | Single DFT bin $X[k]$ |
| **Cost** | $N$ real mults + 1 complex mult |
| **When to use** | Need $M \ll \log_2 N$ specific bins |
| **Core recurrence** | $s[n] = x[n] + 2\cos(\omega_k)\,s[n-1] - s[n-2]$ |
| **Final output** | $X[k] = s[N-1] - e^{-j\omega_k}\,s[N-2]$ |
| **Magnitude only** | $|X[k]|^2 = s_1^2 + s_2^2 - 2\cos(\omega_k)\,s_1 s_2$ (fully real) |
| **Classic use-case** | DTMF decoding (8 fixed frequencies) |
| **Caveat** | Non-integer $k$ is fine — just use the exact $\omega_k$ you want |

The algorithm is also directly applicable to **pitch detection** in a synthesizer context: run Goertzel at the expected harmonic frequencies of each MIDI note and pick the note with highest correlated energy — much cheaper than a full FFT when you're tracking a single voice.